### Spam Ham Classification using BOW

In [1]:
import pandas as pd

In [2]:
messages = pd.read_csv('../Bag of Words/SMSSpamCollection.csv')

In [3]:
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5569,spam,This is the 2nd time we have tried 2 contact u...
5570,ham,Will ü b going to esplanade fr home?
5571,ham,"Pity, * was in mood for that. So...any other s..."
5572,ham,The guy did some bitching but I acted like i'd...


In [4]:
# data cleaning

import re
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
ps = PorterStemmer()

In [5]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\om_da\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
corpus = []
for i in range(len(messages)):
    # remove special characters
    review = re.sub('^[a-zA-Z]', ' ', messages['message'][i])
    # lower all the case
    review = review.lower()
    # convert it to list of words
    review = review.split()
    # if not present in stopwords, apply stemming to it
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [7]:
corpus

['jurong point, crazy.. avail bugi n great world la e buffet... cine got amor wat...',
 'k lar... joke wif u oni...',
 "ree entri 2 wkli comp win fa cup final tkt 21st may 2005. text fa 87121 receiv entri question(std txt rate)t&c' appli 08452810075over18'",
 'dun say earli hor... u c alreadi say...',
 'ah think goe usf, live around though',
 "reemsg hey darl 3 week' word back! like fun still? tb ok! xxx std chg send, £1.50 rcv",
 'ven brother like speak me. treat like aid patent.',
 "per request 'mell mell (oru minnaminungint nurungu vettam)' set callertun callers. press *9 copi friend callertun",
 'inner!! valu network custom select receivea £900 prize reward! claim call 09061701461. claim code kl341. valid 12 hour only.',
 'ad mobil 11 month more? u r entitl updat latest colour mobil camera free! call mobil updat co free 08002986030',
 "'m gonna home soon want talk stuff anymor tonight, k? cri enough today.",
 'ix chanc win cash! 100 20,000 pound txt> csh11 send 87575. cost 150p/day

### Create Bag of Words

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

In [9]:
cv = CountVectorizer(max_features=2500, ngram_range=(1, 2))

In [10]:
# independant features
x = cv.fit_transform(corpus).toarray()

In [11]:
x

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5574, 2500))

In [12]:
cv.vocabulary_

{'point': np.int64(1704),
 'avail': np.int64(271),
 'great': np.int64(942),
 'world': np.int64(2445),
 'la': np.int64(1168),
 'cine': np.int64(482),
 'got': np.int64(933),
 'wat': np.int64(2359),
 'lar': np.int64(1179),
 'joke': np.int64(1137),
 'wif': np.int64(2409),
 'ree': np.int64(1791),
 'entri': np.int64(735),
 'wkli': np.int64(2432),
 'comp': np.int64(527),
 'win': np.int64(2414),
 'cup': np.int64(581),
 'final': np.int64(807),
 'may': np.int64(1325),
 'text': np.int64(2116),
 'receiv': np.int64(1783),
 'question': np.int64(1750),
 'std': np.int64(2037),
 'txt': np.int64(2227),
 'rate': np.int64(1763),
 'appli': np.int64(232),
 'ree entri': np.int64(1793),
 'std txt': np.int64(2038),
 'txt rate': np.int64(2231),
 'rate appli': np.int64(1764),
 'dun': np.int64(686),
 'say': np.int64(1880),
 'earli': np.int64(695),
 'alreadi': np.int64(203),
 'ah': np.int64(184),
 'think': np.int64(2135),
 'goe': np.int64(914),
 'usf': np.int64(2300),
 'live': np.int64(1244),
 'around': np.int64(2

In [13]:
# output features
y = pd.get_dummies(messages['label'])

In [14]:
y

,ham,spam
0,True,False
1,True,False
2,False,True
3,True,False
4,True,False
...,...,...
5569,False,True
5570,True,False
5571,True,False
5572,True,False


In [15]:
y = y.iloc[:,0].values

In [16]:
y

array([ True,  True, False, ...,  True,  True,  True], shape=(5574,))

In [17]:
# train test split
from sklearn.model_selection import train_test_split

In [18]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [19]:
from sklearn.naive_bayes import MultinomialNB

In [20]:
spam_detect_model = MultinomialNB().fit(x_train, y_train)

In [21]:
spam_detect_model

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [23]:
y_pred = spam_detect_model.predict(x_test)

In [24]:
from sklearn.metrics import accuracy_score, classification_report

In [25]:
accuracy_score(y_test, y_pred)

0.9811659192825112

In [27]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.95      0.92      0.93       161
        True       0.99      0.99      0.99       954

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115

